In [31]:
#import pandas as pd
import numpy as np
import os


# --- 0. Merging Raw Magic Leaf Text Files ---
file_paths = [
    "C:/Users/phill/Desktop/AI_MODEL/Magic leaf/MAGIC LEAF-31.5C-DARK.txt",
    "C:/Users/phill/Desktop/AI_MODEL/Magic leaf/MAGIC LEAF-31.5C-LIGHT.txt",
    "C:/Users/phill/Desktop/AI_MODEL/Magic leaf/MAGIC LEAF-50C-DARK.txt",
    "C:/Users/phill/Desktop/AI_MODEL/Magic leaf/MAGIC LEAF-50C-LIGHT.txt",
    "C:/Users/phill/Desktop/AI_MODEL/Magic leaf/MAGIC LEAF-70C-DARK.txt",
    "C:/Users/phill/Desktop/AI_MODEL/Magic leaf/MAGIC LEAF-70C-LIGHT.txt",   
]

In [32]:
import pandas as pd
dfs = []
missing = []
for path in file_paths:
    if os.path.exists(path):
        # Based on raw file preview: Col 0 is Voltage, Col 1 is Current
        temp_df = pd.read_csv(path, sep=None, engine='python', names=['Voltage', 'Current'], skiprows=1)

        filename = os.path.basename(path)
        temp_df['Temperature'] = float(filename.split('-')[1].replace('C', ''))
        temp_df['Condition'] = filename.split('-')[2].replace('.txt', '')

        # Create a synthetic Time column based on row index for temporal tracking
        temp_df['Time'] = np.arange(len(temp_df))
        dfs.append(temp_df)
    else:
        missing.append(path)

if missing:
    print(f"WARNING: {len(missing)} file(s) not found and were skipped:")
    for p in missing:
        print(f"  - {p}")

assert dfs, "No input files were found — check file_paths before continuing."
data = pd.concat(dfs, ignore_index=True)
data['Voltage'] = pd.to_numeric(data['Voltage'], errors='coerce')
data['Current'] = pd.to_numeric(data['Current'], errors='coerce')

print(f"Loaded {len(dfs)} of {len(file_paths)} files, {len(data)} total rows.")

Loaded 6 of 6 files, 736 total rows.


In [33]:

# Export the processed dataset to CSV
data.to_csv('merged_magic_leaf_data.csv', index=False)
print("Merge complete. Voltage and Current columns correctly mapped from raw files.")
data.head(10000)


Merge complete. Voltage and Current columns correctly mapped from raw files.


,Voltage,Current,Temperature,Condition,Time
0,-0.130150,0.000032,31.5,DARK,0
1,-0.122746,0.000029,31.5,DARK,1
2,-0.122135,0.000027,31.5,DARK,2
3,-0.122621,0.000025,31.5,DARK,3
4,-0.111118,0.000023,31.5,DARK,4
...,...,...,...,...,...
731,0.676016,-0.014616,70.0,LIGHT,117
732,0.675829,-0.014919,70.0,LIGHT,118
733,0.680263,-0.015379,70.0,LIGHT,119
734,0.692008,-0.015957,70.0,LIGHT,120


**Advanced Feature Extraction**

This section details the creation of new, physics-informed features from the raw sensor data. These features are crucial for providing a comprehensive understanding of the 'Magic Leaf' panel's performance and for building a more robust predictive model.


In [34]:
# --- Diagnose and correct Current's sign convention automatically ---
# Some source-meter setups record Current as negative under one condition
# (commonly flips between LIGHT vs DARK depending on wiring convention).
# Diagnose per (Temperature, Condition) group, and flip any group whose
# median Current is negative so downstream features (current_density, Power,
# efficiency) don't need repeated .abs() patches later in the notebook.

print("Current min/max/median by Temperature and Condition (before correction):")
print(data.groupby(['Temperature', 'Condition'])['Current'].agg(['min', 'max', 'median']))

def _fix_sign(group):
    if group['Current'].median() < 0:
        group['Current'] = group['Current'] * -1
    return group

data = data.groupby(['Temperature', 'Condition'], group_keys=False).apply(_fix_sign)

print("\nCurrent min/max/median by Temperature and Condition (after correction):")
print(data.groupby(['Temperature', 'Condition'])['Current'].agg(['min', 'max', 'median']))

Current min/max/median by Temperature and Condition (before correction):
                            min       max    median
Temperature Condition                              
31.5        DARK      -0.064967  0.000032 -0.000646
            LIGHT     -0.217189  0.099720  0.065177
50.0        DARK      -0.069240  0.000077 -0.000835
            LIGHT     -0.223692  0.095492  0.053866
70.0        DARK      -0.015518  0.000177 -0.001168
            LIGHT     -0.016287  0.033699  0.009029

Current min/max/median by Temperature and Condition (after correction):
                            min       max    median
Temperature Condition                              
31.5        DARK      -0.000032  0.064967  0.000646
            LIGHT     -0.217189  0.099720  0.065177
50.0        DARK      -0.000077  0.069240  0.000835
            LIGHT     -0.223692  0.095492  0.053866
70.0        DARK      -0.000177  0.015518  0.001168
            LIGHT     -0.016287  0.033699  0.009029


C:\Users\phill\AppData\Local\Temp\ipykernel_18072\1856620807.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data = data.groupby(['Temperature', 'Condition'], group_keys=False).apply(_fix_sign)



### 1. `angle_deg` (Synthetic Incidence Angle)

*   **Reason**: To simulate the angle at which light might strike the panel, allowing us to incorporate the effect of incidence angle on current generation, which is a key factor in solar energy systems.
*   **Formula**: A linearly increasing sequence from 0 to 90 degrees across the dataset length.
    

### 2. `irradiance_wm2` (Simulated Solar Irradiance)

*   **Reason**: To represent the intensity of light falling on the panel. This is a primary driver of current generation. It's set based on whether the 'Condition' is 'LIGHT' or 'DARK'.
*   **Formula**: If `Condition` is 'LIGHT', then 1000 W/m² (a typical value for full sunlight); otherwise, 0 W/m².
  

In [35]:
# 1. Generate synthetic angle per experiment group
data['angle_deg'] = data.groupby(['Temperature', 'Condition']).cumcount()
data['angle_deg'] = data.groupby(['Temperature', 'Condition'])['angle_deg'].transform(
    lambda x: np.linspace(0, 90, len(x))
)

# 2. Simulate solar irradiance
data['irradiance_wm2'] = np.where(data['Condition'] == 'LIGHT', 1000, 0)

### 3. `Condition_Encoded` (Numerical Light Condition)

*   **Reason**: Machine learning models typically require numerical input. This feature converts the categorical 'Condition' (LIGHT/DARK) into numerical values.

*   **Technique**: Label Encoding is used for assigning an integer to each unique category.
    *   `Condition_Encoded = LabelEncoder().fit_transform(data['Condition'])`

In [36]:
# 3. Encode categorical light conditions
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data['Condition_Encoded'] = le.fit_transform(data['Condition'])


### 4. `Cos_Angle_Dynamic` (Cosine of Incidence Angle)

*   **Reason**: Derived from Lambert's Cosine Law, which states that the effective irradiance on a surface is proportional to the cosine of the angle of incidence. This directly impacts how much light is absorbed and thus the generated current.

*   **Formula**: $Cos\_Angle\_Dynamic = \cos(\text{angle\_deg in radians})$
  

In [37]:
# 4. Physics Feature: Cosine of incidence angle (Lambert's Law)
data['Cos_Angle_Dynamic'] = np.cos(np.radians(data['angle_deg']))


### 5. `current_density` (Current per Unit Area)

*   **Reason**: Normalizes the current output by the device's area, making the metric independent of the specific panel size and comparable across different experimental setups. (Assuming `area_cm2` is 1.0 for this dataset).

*   **Formula**: $\text{Current Density} = \frac{|\text{Current}|}{\text{Area}}$
  

In [38]:
# 5. Feature: Current Density
area_cm2 = 1.0
data['current_density'] = data['Current'] / area_cm2

### 6. `Power` (Electrical Power Output)

*   **Reason**: Represents the rate at which electrical energy is generated, a fundamental performance metric for any energy-producing device.

*   **Formula**: $\text{Power} = |\text{Voltage}| \times |\text{Current}|$
   

In [39]:
# 6. Feature: Power calculation (P = V * I)
data['Power'] = (data['Current'] * data['Voltage'])

### 7. `efficiency` (Energy Conversion Efficiency)

*   **Reason**: Quantifies how effectively the 'Magic Leaf' converts the incident light energy (`irradiance_wm2`) into useful electrical power (`Power`). It's a critical indicator of device performance.

*   **Formula**: $\eta = \frac{\text{Electrical Power Output}}{\text{Incident Light Power}}$

    *   Incident Light Power (mW) = `irradiance_wm2` (W/m²) * `area_cm2` (m²) * 1000 (mW/W)

    *   `efficiency = Power / Incident Light Power`

In [40]:
# 7. Efficiency Calculation using Absolute Power
input_power_mw = data['irradiance_wm2'] * (area_cm2 * 0.0001) * 1000

data['efficiency'] = np.where(input_power_mw > 0, data['Power'] / input_power_mw, 0)


In [41]:
data.head(15)

,Voltage,Current,Temperature,Condition,Time,angle_deg,irradiance_wm2,Condition_Encoded,Cos_Angle_Dynamic,current_density,Power,efficiency
0,-0.130150,-3.165910e-05,31.5,DARK,0,0.000000,0,0,1.000000,-3.165910e-05,4.120445e-06,0.0
1,-0.122746,-2.935400e-05,31.5,DARK,1,0.743802,0,0,0.999916,-2.935400e-05,3.603073e-06,0.0
2,-0.122135,-2.715520e-05,31.5,DARK,2,1.487603,0,0,0.999663,-2.715520e-05,3.316600e-06,0.0
3,-0.122621,-2.485660e-05,31.5,DARK,3,2.231405,0,0,0.999242,-2.485660e-05,3.047932e-06,0.0
4,-0.111118,-2.259930e-05,31.5,DARK,4,2.975207,0,0,0.998652,-2.259930e-05,2.511187e-06,0.0
5,-0.109382,-2.038400e-05,31.5,DARK,5,3.719008,0,0,0.997894,-2.038400e-05,2.229635e-06,0.0
6,-0.108399,-1.816900e-05,31.5,DARK,6,4.462810,0,0,0.996968,-1.816900e-05,1.969496e-06,0.0
7,-0.097118,-1.581350e-05,31.5,DARK,7,5.206612,0,0,0.995874,-1.581350e-05,1.535773e-06,0.0
8,-0.092548,-1.366550e-05,31.5,DARK,8,5.950413,0,0,0.994612,-1.366550e-05,1.264709e-06,0.0
9,-0.093966,-1.133930e-05,31.5,DARK,9,6.694215,0,0,0.993182,-1.133930e-05,1.065510e-06,0.0



### 8. IV Metrics (Isc, Voc, Pmax, Fill Factor)

In the context of solar energy, several key metrics derived from the current-voltage (IV) curve are crucial for understanding and characterizing the performance of a photovoltaic device like our 'Magic Leaf' panel. These metrics provide insights into the panel's efficiency and power generation capabilities.

In [42]:
#  8. Define IV metrics calculation function
def calculate_iv_metrics(group):
    v = group['Voltage'].values
    i = group['Current'].values
    p = group['Power'].values

    # Isc = Current where Voltage is closest to 0
    idx_v0 = np.abs(v).argmin()
    isc = abs(i[idx_v0])

    # Voc = Voltage where Current is closest to 0
    idx_i0 = np.abs(i).argmin()
    voc = abs(v[idx_i0])

    # Fill Factor calculation (Pmax / (Isc * Voc))
    pmax = p.max()
    ff = pmax / (isc * voc) if (isc * voc) != 0 else 0

    return pd.Series({'Isc': isc, 'Voc': voc, 'Pmax': pmax, 'Fill_Factor': ff})

#### 1.  `Short-Circuit Current (Isc) `
*   **Definition**: `Isc` is the maximum current delivered by the cell when the voltage across it is zero (i.e., when the cell is short-circuited). It represents the highest current the device can produce under a given illumination and temperature.
*   **Calculation Approach**: We approximate `Isc` by taking the absolute current value at the point where the absolute voltage is at its minimum within each experimental group.

#### 2.  `Open-Circuit Voltage (Voc) `
*   **Definition**: `Voc` is the maximum voltage across the cell when no current is flowing through it (i.e., when the cell is open-circuited). It represents the maximum potential difference the device can generate.
*   **Calculation Approach**: We approximate `Voc` by taking the absolute voltage value at the point where the absolute current is at its minimum within each experimental group.

#### 3.  `Maximum Power (Pmax) `
*   **Definition**: `Pmax` is the maximum power that the device can deliver to an external load. It is the product of the current and voltage at the point on the IV curve where the power output is highest (Maximum Power Point or MPP).
*   **Calculation Approach**: We directly calculate `Pmax` by finding the maximum value of the `Power` feature (which is `Voltage * Current`) within each experimental group.

#### 4.  ` Factor (FF) `
*   **Definition**: `Fill Factor` is a measure of the quality of the solar cell. It is the ratio of the maximum power to the product of the open-circuit voltage and the short-circuit current.

*   **Formula**: $FF = \frac{P_{max}}{I_{sc} \times V_{oc}}$

*   **Interpretation**: A higher fill factor indicates a more rectangular IV curve, implying a higher quality device that can deliver more power efficiently. It quantifies how well the cell's IV curve approximates an ideal rectangular shape.

*   **Calculation Approach**: We compute the Fill Factor using the derived `Pmax`, `Isc`, and `Voc` values, with a safeguard against division by zero.

In [43]:
# Drop existing IV columns to prevent merge duplicates (if they exist from a previous run)
columns_to_drop = ['Isc', 'Voc', 'Pmax', 'Fill_Factor']
for col in columns_to_drop:
    if col in data.columns:
        data = data.drop(columns=[col])


In [44]:

# Apply IV metrics calculations to each experimental run
iv_metrics = data.groupby(['Temperature', 'Condition']).apply(calculate_iv_metrics, include_groups=False).reset_index()
data = data.merge(iv_metrics, on=['Temperature', 'Condition'])

print("All feature engineering steps completed")
data.head()

All feature engineering steps completed


,Voltage,Current,Temperature,Condition,Time,angle_deg,irradiance_wm2,Condition_Encoded,Cos_Angle_Dynamic,current_density,Power,efficiency,Isc,Voc,Pmax,Fill_Factor
0,-0.130150,-0.000032,31.5,DARK,0,0.000000,0,0,1.000000,-0.000032,0.000004,0.0,0.000034,0.060603,0.031519,15197.24329
1,-0.122746,-0.000029,31.5,DARK,1,0.743802,0,0,0.999916,-0.000029,0.000004,0.0,0.000034,0.060603,0.031519,15197.24329
2,-0.122135,-0.000027,31.5,DARK,2,1.487603,0,0,0.999663,-0.000027,0.000003,0.0,0.000034,0.060603,0.031519,15197.24329
3,-0.122621,-0.000025,31.5,DARK,3,2.231405,0,0,0.999242,-0.000025,0.000003,0.0,0.000034,0.060603,0.031519,15197.24329
4,-0.111118,-0.000023,31.5,DARK,4,2.975207,0,0,0.998652,-0.000023,0.000003,0.0,0.000034,0.060603,0.031519,15197.24329


In [45]:
# Calculate mean efficiency per group and combine with the IV metrics report.

mean_efficiency_per_group = data.groupby(['Temperature', 'Condition'])['efficiency'].mean().reset_index()
efficiency_summary = iv_metrics.merge(mean_efficiency_per_group, on=['Temperature', 'Condition'])

In [46]:
print("--- Final Magic Leaf Performance Report ---")
display(efficiency_summary.rename(columns={'efficiency': 'Mean_Efficiency'}))

--- Final Magic Leaf Performance Report ---


,Temperature,Condition,Isc,Voc,Pmax,Fill_Factor,Mean_Efficiency
0,31.5,DARK,0.000034,0.060603,0.031519,15197.243290,0.000000
1,31.5,LIGHT,0.095642,0.381258,0.016663,0.456954,-0.000165
2,50.0,DARK,0.000109,0.065259,0.033474,4704.881431,0.000000
3,50.0,LIGHT,0.090319,0.374125,0.014919,0.441500,-0.000179
4,70.0,DARK,0.000136,0.055556,0.010913,1447.078915,0.000000
5,70.0,LIGHT,0.023036,0.394933,0.002429,0.266975,-0.000013


 **MACHINE LEARNING MODEL DEPLOYMENT AND EVALUATION**

In [47]:
data.head(1000)

,Voltage,Current,Temperature,Condition,Time,angle_deg,irradiance_wm2,Condition_Encoded,Cos_Angle_Dynamic,current_density,Power,efficiency,Isc,Voc,Pmax,Fill_Factor
0,-0.130150,-0.000032,31.5,DARK,0,0.000000,0,0,1.000000e+00,-0.000032,0.000004,0.000000,0.000034,0.060603,0.031519,15197.243290
1,-0.122746,-0.000029,31.5,DARK,1,0.743802,0,0,9.999157e-01,-0.000029,0.000004,0.000000,0.000034,0.060603,0.031519,15197.243290
2,-0.122135,-0.000027,31.5,DARK,2,1.487603,0,0,9.996630e-01,-0.000027,0.000003,0.000000,0.000034,0.060603,0.031519,15197.243290
3,-0.122621,-0.000025,31.5,DARK,3,2.231405,0,0,9.992417e-01,-0.000025,0.000003,0.000000,0.000034,0.060603,0.031519,15197.243290
4,-0.111118,-0.000023,31.5,DARK,4,2.975207,0,0,9.986521e-01,-0.000023,0.000003,0.000000,0.000034,0.060603,0.031519,15197.243290
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
731,0.676016,-0.014616,70.0,LIGHT,117,87.024793,1000,1,5.190382e-02,-0.014616,-0.009881,-0.000099,0.023036,0.394933,0.002429,0.266975
732,0.675829,-0.014919,70.0,LIGHT,118,87.768595,1000,1,3.893552e-02,-0.014919,-0.010082,-0.000101,0.023036,0.394933,0.002429,0.266975
733,0.680263,-0.015379,70.0,LIGHT,119,88.512397,1000,1,2.596066e-02,-0.015379,-0.010462,-0.000105,0.023036,0.394933,0.002429,0.266975
734,0.692008,-0.015957,70.0,LIGHT,120,89.256198,1000,1,1.298142e-02,-0.015957,-0.011043,-0.000110,0.023036,0.394933,0.002429,0.266975


### Feature and Target Selection

*   **Target Variable (`y`)**: Our target was `Current`, representing the electrical current generated by the panel.

*   **Features (`X`)**: We carefully selected a set of features that we engineered based on physics and observational data. These included:
     *   `Voltage`, `Temperature`, `Condition_Encoded`, `Time`, `Cos_Angle_Dynamic`.

This rich set of features, combining raw measurements with physics-informed derivations, allowed the model to learn complex patterns in the data.

In [48]:

features = ['Voltage', 'Temperature', 'Condition_Encoded', 'Time', 'Cos_Angle_Dynamic']
X = data[features]
y = data['Current']

In [49]:
#checking out the features encoded data
data[features].head(10)

,Voltage,Temperature,Condition_Encoded,Time,Cos_Angle_Dynamic
0,-0.130150,31.5,0,0,1.000000
1,-0.122746,31.5,0,1,0.999916
2,-0.122135,31.5,0,2,0.999663
3,-0.122621,31.5,0,3,0.999242
4,-0.111118,31.5,0,4,0.998652
5,-0.109382,31.5,0,5,0.997894
6,-0.108399,31.5,0,6,0.996968
7,-0.097118,31.5,0,7,0.995874
8,-0.092548,31.5,0,8,0.994612
9,-0.093966,31.5,0,9,0.993182


In [50]:
#checking out the target data
y

0     -0.000032
1     -0.000029
2     -0.000027
3     -0.000025
4     -0.000023
         ...   
731   -0.014616
732   -0.014919
733   -0.015379
734   -0.015957
735   -0.016287
Name: Current, Length: 736, dtype: float64

### Dataset Splitting: Training and Testing Sets

To evaluate the performance of a machine learning model, it's crucial to split the dataset into two main parts: a training set and a testing set. The model learns from the training data and is then evaluated on the unseen testing data to assess its generalization capabilities.

Given a dataset $D$ with $N$ samples, we typically split it into:

1.  **Training Set ($D_{train}$)**: Used to train the model. It contains a proportion $p$ of the original data. The number of samples in the training set is $N_{train} = p \times N$.

2.  **Testing Set ($D_{test}$)**: Used to evaluate the trained model. It contains the remaining proportion $(1-p)$ of the data. The number of samples in the testing set is $N_{test} = (1-p) \times N$.

In our case, we used `test_size=0.2`, meaning $p = 0.8$ for the training set and $1-p = 0.2$ for the testing set.

Let $X$ represent the features and $y$ represent the target variable.

$$D = \{(X_i, y_i)\}_{i=1}^{N}$$

Split into:

$$D_{train} = \{(X_i, y_i) \mid i \in \text{train indices}\}$$

$$D_{test} = \{(X_i, y_i) \mid i \in \text{test indices}\}$$

Where the union of train indices and test indices forms the complete set of indices, and their intersection is empty. The `random_state` parameter ensures reproducibility of the split.

In [51]:
from sklearn.model_selection import train_test_split

# Split into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


Model Building and Evaluation


Our primary objective is to accurately predict the `Current` output of the 'Magic Leaf' panel. This is a regression problem, where we aim to forecast a continuous numerical value. To achieve this, we selected a **Random Forest Regressor**, a powerful and versatile machine learning algorithm.

### Random Forest Regressor 

A Random Forest is an ensemble learning method for regression (and classification) that operates by constructing a multitude of decision trees during training and outputting the mean prediction (for regression) or the mode of the classes (for classification) of the individual trees.

Mathematically, for a regression problem, if we have $B$ decision trees in the forest, and each tree $T_b(x)$ provides a prediction for an input $x$, the final Random Forest prediction $\hat{f}_{RF}(x)$ is the average of the predictions from all individual trees:

$$\hat{f}_{RF}(x) = \frac{1}{B} \sum_{b=1}^{B} T_b(x)$$

Each tree $T_b$ is grown on a bootstrap sample of the training data. Additionally, when building each tree, a random subset of features is considered at each split, which helps to decorrelate the trees and reduce variance.

In [52]:
from sklearn.ensemble import RandomForestRegressor

# Deploy Random Forest Regressor
#mathematical formula: Randfom forest regresor model 

model = RandomForestRegressor(n_estimators=100, random_state=42) 
model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

### The Evaluation Journey

After training the Random Forest Regressor, we evaluated its performance on unseen data (the test set) using two key metrics: **R-squared (R²)** and **Mean Absolute Error (MAE)**.

#### 1. R-Squared Score (R²)

*   **Reasoning**: R-squared is a statistical measure that represents the proportion of the variance in the dependent variable (Current) that can be predicted from the independent variables (our features). A higher R-squared value indicates a better fit of the model to the data. In the context of Leave-One-Condition-Out cross-validation, it tells us how well the model generalizes to entirely unseen experimental conditions.

*   **Manual Calculation Formula**:
    $R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$
    Where:
    
    *   $y_i$ is the actual value.
    *   $\hat{y}_i$ is the predicted value.
    *   $\bar{y}$ is the mean of the actual values.
    *   The numerator is the Residual Sum of Squares (RSS).
    *   The denominator is the Total Sum of Squares (TSS).

#### 2. Mean Absolute Error (MAE)

*   **Reasoning**: MAE measures the average magnitude of the errors in a set of predictions, without considering their direction. It tells us, on average, how much our predictions deviate from the actual values. A lower MAE indicates a more accurate model.

*   **Manual Calculation Formula**:

    $MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$
    Where:
    *   $n$ is the number of data points.
    *   $y_i$ is the actual value.
    *   $\hat{y}_i$ is the predicted value.

#### 1. R-Squared Score (R²)

*   **Reasoning**: R-squared is a statistical measure that represents the proportion of the variance in the dependent variable (Current) that can be predicted from the independent variables (our features). A higher R-squared value indicates a better fit of the model to the data.
*   **Our Finding**: An R-squared score of approximately **0.997650** is exceptionally high. This suggests that nearly 99.8% of the variability in the 'Magic Leaf' panel's current output can be explained by our model's features. This is a strong indicator that our model is capturing the underlying physical processes very well.

*   **Manual Calculation Formula**:
    $R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$
    Where:
    
    *   $y_i$ is the actual value.
    *   $\hat{y}_i$ is the predicted value.
    *   $\bar{y}$ is the mean of the actual values.
    *   The numerator is the Residual Sum of Squares (RSS).
    *   The denominator is the Total Sum of Squares (TSS).

#### 2. Mean Absolute Error (MAE)

*   **Reasoning**: MAE measures the average magnitude of the errors in a set of predictions, without considering their direction. It tells us, on average, how much our predictions deviate from the actual values. A lower MAE indicates a more accurate model.

*   **Our Finding**: A Mean Absolute Error of approximately **1.406771e-03 mA** (which is about 0.000844 mA) is extremely low. Given the typical range of current values in our dataset, this indicates that the model's predictions are very close to the actual measured current, with an average absolute difference of less than a milliampere. This reinforces the high accuracy suggested by the R-squared score.
*   **Manual Calculation Formula**:

    $MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$
    Where:
    *   $n$ is the number of data points.
    *   $y_i$ is the actual value.
    *   $\hat{y}_i$ is the predicted value.

In [53]:
# Feature Importance
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print("\n--- Feature Importance Report ---")
print(importances)



--- Feature Importance Report ---
Voltage              0.419355
Temperature          0.296921
Cos_Angle_Dynamic    0.178959
Condition_Encoded    0.081703
Time                 0.023062
dtype: float64


# --- Leave-one-condition-out evaluation ---
Holds out one full (Temperature, Condition) experiment at a time, so the
model is tested on a genuinely unseen condition rather than on rows
adjacent to ones it already saw within the same sweep.

In [54]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error


loo_results = []
for temp, cond in data[['Temperature', 'Condition']].drop_duplicates().values:
    test_mask = (data['Temperature'] == temp) & (data['Condition'] == cond)
    X_tr, X_te = X[~test_mask], X[test_mask]
    y_tr, y_te = y[~test_mask], y[test_mask]

    m = RandomForestRegressor(n_estimators=100, random_state=42)
    m.fit(X_tr, y_tr)
    pred = m.predict(X_te)

    loo_results.append({
        'held_out': f"{temp}C-{cond}",
        'r2': r2_score(y_te, pred),
        'mae': mean_absolute_error(y_te, pred)
    })

loo_df = pd.DataFrame(loo_results)
print("--- Leave-One-Condition-Out Results (the number to actually trust) ---")
display(loo_df)
print(f"\nMean R2:  {loo_df['r2'].mean():.4f}")
print(f"Mean MAE: {loo_df['mae'].mean():.6e} mA")

# Conventional random 80/20 split, kept for comparison only -- rows within
# the same sweep are highly correlated, so this number tends to look better
# than the model's true generalization performance.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

--- Leave-One-Condition-Out Results (the number to actually trust) ---


,held_out,r2,mae
0,31.5C-DARK,0.787055,0.007437
1,31.5C-LIGHT,0.988323,0.008192
2,50.0C-DARK,0.989020,0.001193
3,50.0C-LIGHT,0.975026,0.009832
4,70.0C-DARK,-4.250451,0.006148
5,70.0C-LIGHT,-12.631872,0.047064



Mean R2:  -2.1905
Mean MAE: 1.331125e-02 mA


In [55]:
from sklearn.metrics import r2_score, mean_absolute_error

# Predictions and Metrics
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("--- Model Evaluation Report ---")
print(f"\nMean R Score: {r2:.6f}")
print(f"Mean Absolute Error: {mae:.6e} mA")

print(f"\nFor mean leave-one-condition-out")
R2 =  f"{loo_df['r2'].mean():.4f}"
MAE = f"{loo_df['mae'].mean():.6e} mA"
print(f"\nMean R Score: {R2}")
print(f"Mean Absolute Error: {MAE}")

--- Model Evaluation Report ---

Mean R Score: 0.997609
Mean Absolute Error: 1.375512e-03 mA

For mean leave-one-condition-out

Mean R Score: -2.1905
Mean Absolute Error: 1.331125e-02 mA


In [56]:
top_features = importances.index[:3].tolist()

print(f"Top features:{top_features}")

Top features:['Voltage', 'Temperature', 'Cos_Angle_Dynamic']


The feature importance report shows that Voltage, Temperature, and
Cos_Angle_Dynamic are the most influential predictors of Current, which is consistent
with their direct relationship to the panel's IV behavior.

Earlier versions of this model included current_density, Power, and efficiency as
predictors. On inspection these are algebraic functions of Current itself, so their
inclusion caused the model to substantially reconstruct the target from derived
versions of itself rather than learn from independent conditions. They were removed
from the feature set.

Evaluated with a conventional random 80/20 split, the model reports R2 = 0.9976,
MAE = 1.375512e-03 mA. Because adjacent rows within the same IV sweep are highly
correlated, a random split allows near-duplicate points into both train and test
sets, which can inflate this score.

Evaluated instead with leave-one-condition-out cross-validation -- holding out each
full (Temperature, Condition) experiment in turn -- the model achieves a mean R2 of
-2.1905 and mean MAE of 1.331125e-02 mA across the six
held-out conditions. This is the more honest measure of how well the model
generalizes to a genuinely unseen experimental condition, and is the number that
should be reported as the model's real performance

## Deployment: Predicting Current from Live Panel Readings

`Cos_Angle_Dynamic` is dropped for deployment. It's built from each row's *position
within a finished historical experiment* (`groupby(...).cumcount()` scaled to a
0-90 degree ramp), so it can only be computed retroactively over a completed batch
of readings -- a single live reading streaming in from the panel has no such
position to compute it from. The deployment feature set below uses only values a
live reading actually provides: Voltage, Temperature, Condition, and Time.

In [57]:
import joblib

# Deployment feature set -- everything a live panel reading actually provides.
deploy_features = ['Voltage', 'Temperature', 'Condition_Encoded', 'Time']
X_deploy = data[deploy_features]
y_deploy = data['Current']

deploy_loo_results = []
for temp, cond in data[['Temperature', 'Condition']].drop_duplicates().values:
    test_mask = (data['Temperature'] == temp) & (data['Condition'] == cond)
    X_tr, X_te = X_deploy[~test_mask], X_deploy[test_mask]
    y_tr, y_te = y_deploy[~test_mask], y_deploy[test_mask]

    m = RandomForestRegressor(n_estimators=100, random_state=42)
    m.fit(X_tr, y_tr)
    pred = m.predict(X_te)
    deploy_loo_results.append({
        'held_out': f"{temp}C-{cond}",
        'r2': r2_score(y_te, pred),
        'mae': mean_absolute_error(y_te, pred)
    })

deploy_loo_df = pd.DataFrame(deploy_loo_results)
print("--- Leave-One-Condition-Out (deployment feature set, no Cos_Angle_Dynamic) ---")
display(deploy_loo_df)
print(f"Mean R2:  {deploy_loo_df['r2'].mean():.4f}")
print(f"Mean MAE: {deploy_loo_df['mae'].mean():.6e} mA")

# Train the final deployed model on ALL historical data (no holdout -- for
# deployment you want the model to use everything you've got).
final_model = RandomForestRegressor(n_estimators=100, random_state=42)
final_model.fit(X_deploy, y_deploy)

joblib.dump(
    {'model': final_model, 'label_encoder': le, 'features': deploy_features},
    'magic_leaf_current_predictor.joblib'
)
print("Saved: magic_leaf_current_predictor.joblib")

--- Leave-One-Condition-Out (deployment feature set, no Cos_Angle_Dynamic) ---


,held_out,r2,mae
0,31.5C-DARK,0.435037,0.011405
1,31.5C-LIGHT,0.983740,0.009398
2,50.0C-DARK,0.985317,0.001375
3,50.0C-LIGHT,0.971343,0.010648
4,70.0C-DARK,-4.201334,0.006083
5,70.0C-LIGHT,-14.075759,0.049507


Mean R2:  -2.4836
Mean MAE: 1.473602e-02 mA
Saved: magic_leaf_current_predictor.joblib


In [58]:
def predict_current(voltage, temperature, condition, time_index,
                     model_path='magic_leaf_current_predictor.joblib'):
    """
    Predict Current (mA) from a single incoming panel reading.

    Parameters:
        voltage (float): measured voltage (V)
        temperature (float): panel temperature (C)
        condition (str): 'LIGHT' or 'DARK'
        time_index (int): sample index / elapsed step, defined the same way
            as during training (np.arange per historical file)
        model_path (str): path to the saved joblib bundle

    Returns:
        float: predicted Current (mA)
    """
    bundle = joblib.load(model_path)
    model = bundle['model']
    encoder = bundle['label_encoder']
    features = bundle['features']

    condition_encoded = encoder.transform([condition])[0]

    row = pd.DataFrame([{
        'Voltage': voltage,
        'Temperature': temperature,
        'Condition_Encoded': condition_encoded,
        'Time': time_index
    }])[features]

    return model.predict(row)[0]

In [59]:
# Example: a hypothetical live reading
predicted = predict_current(voltage=0.79, temperature=80.0, condition='LIGHT', time_index=120)
print(f"Predicted Current: {predicted:.4f} mA")

Predicted Current: -0.0159 mA
